# 04 · Validación del tablero web

El producto es `index.html`. No requiere Jupyter para uso diario. Esta etapa comprueba los archivos estáticos y ejecuta las pruebas de reglas de importación, calidad y cálculo.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if not (ROOT / "scripts" / "preparar.cjs").exists():
    ROOT = ROOT.parent
assert (ROOT / "scripts" / "preparar.cjs").exists(), "Abre el notebook desde la raíz o notebooks/."
PROCESSED = ROOT / "data" / "processed"
from html.parser import HTMLParser
import subprocess, sys

class Assets(HTMLParser):
    def __init__(self):
        super().__init__()
        self.paths = []
    def handle_starttag(self, tag, attrs):
        d = dict(attrs)
        k = {"script": "src", "link": "href"}.get(tag)
        if k and d.get(k):
            self.paths.append(d[k])

page = Assets()
page.feed((ROOT / "index.html").read_text(encoding="utf-8"))
assert all((ROOT / p).is_file() for p in page.paths)
tests = sorted(p.name for p in (ROOT / "tests").glob("*.test.cjs") if p.name != "sources.test.cjs")
run = subprocess.run(["node", "--test", *[str(ROOT / "tests" / t) for t in tests]], cwd=ROOT, check=True, text=True, capture_output=True)
print(run.stdout)
subprocess.run([sys.executable, str(ROOT / "scripts" / "preparar_sitio.py")], cwd=ROOT, check=True)
assert not any(p.suffix.lower() in {".xlsx", ".xls", ".csv", ".ipynb"} for p in (ROOT / "_site").rglob("*"))
print("Vista local: python -m http.server 8000 --bind 127.0.0.1 --directory _site")

La prueba del flujo completo en navegador y el despliegue de GitHub Pages se explican en el README. El despliegue publica una lista explícita de archivos del sitio; los Excel y las auditorías se quedan fuera.